# grok-001 · T4 冒烟测试（学习向）

> **你在学什么**：确认 Kaggle 的 GPU（常见为 T4×2）能被 PyTorch 看见，并完成一次最小的「CUDA 算子 + 小网络训练」。
>
> **关键概念**：设备 `cuda:0`、fp16 吞吐、训练三步（前向 / 反向 / 更新）。
>
> **不学什么**：正经模型效果、LoRA、评测体系（留给 002+）。

## 建议阅读顺序
1. 打印 GPU 信息 → 建立「设备」直觉  
2. 矩阵乘计时 → 感受 GPU 算力  
3. 小 CNN 几个 epoch → 看 loss 是否下降  


# grok-001-t4-smoke-cnn

> **Slug/title:** `grok-001-t4-smoke-cnn` — T4 smoke: GEMM + tiny CNN on synthetic data


Lightweight GPU job sized for a single **NVIDIA T4**:
1. Verify CUDA device (expect Tesla T4)
2. FP32 GEMM micro-benchmark
3. Train a tiny CNN on synthetic 32×32 data for a few epochs
4. Write results under `/kaggle/working`


In [ ]:
# 【步骤】检查有几张 GPU、名字是否为 Tesla T4
import json, time, platform
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("torch", torch.__version__)
print("python", platform.python_version())
print("cuda_available", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA not available — GPU not attached"
print("device_count", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"device[{i}]", torch.cuda.get_device_name(i),
          "cap", torch.cuda.get_device_capability(i),
          "mem_gb", round(torch.cuda.get_device_properties(i).total_memory / 1e9, 2))
device = torch.device("cuda:0")
print("using", device)


In [ ]:
# FP32 GEMM micro-benchmark (fits easily on T4 16GB)
N = 4096
a = torch.randn(N, N, device=device)
b = torch.randn(N, N, device=device)
for _ in range(3):
    c = a @ b
torch.cuda.synchronize()
t0 = time.perf_counter()
iters = 10
for _ in range(iters):
    c = a @ b
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
flops = 2 * (N ** 3) * iters
tflops = flops / elapsed / 1e12
print(f"GEMM {N}x{N} x{iters}: {elapsed:.3f}s, ~{tflops:.2f} TFLOPS (FP32)")
gemm = {"n": N, "iters": iters, "seconds": elapsed, "tflops_fp32": tflops}


In [ ]:
# 【步骤】固定随机种子，保证数据与实验可复现
# Tiny CNN training on synthetic data (no external download)
class TinyCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
n_train, n_val = 4096, 512  # 划分验证集：调参用，不等于最终测试
x_train = torch.randn(n_train, 3, 32, 32)
y_train = torch.randint(0, 10, (n_train,))
x_val = torch.randn(n_val, 3, 32, 32)
y_val = torch.randint(0, 10, (n_val,))

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True, num_workers=0)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=256, shuffle=False, num_workers=0)

model = TinyCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

history = []
epochs = 5
t_train0 = time.perf_counter()
for epoch in range(1, epochs + 1):
    model.train()
    total_loss, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    train_loss = total_loss / n

    model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(dim=1)
            correct += (pred == yb).sum().item()
            n += yb.size(0)
    val_acc = correct / n
    history.append({"epoch": epoch, "train_loss": train_loss, "val_acc": val_acc})
    print(f"epoch {epoch}/{epochs}  train_loss={train_loss:.4f}  val_acc={val_acc:.3f}")

train_seconds = time.perf_counter() - t_train0
print(f"train_time_s={train_seconds:.2f}")


In [ ]:
# 【步骤】检查有几张 GPU、名字是否为 Tesla T4
# Persist summary
out = {
    "notebook": "grok-001",
    "task": "t4_smoke_cnn",
    "cuda_available": True,
    "device_name": torch.cuda.get_device_name(0),
    "device_count": torch.cuda.device_count(),
    "gemm": gemm,
    "train_seconds": train_seconds,
    "history": history,
    "params": sum(p.numel() for p in model.parameters()),
}
path = Path("/kaggle/working/grok001_results.json")
path.write_text(json.dumps(out, indent=2))
print("wrote", path)
print(json.dumps(out, indent=2))
torch.save({"model": model.state_dict(), "history": history}, "/kaggle/working/tiny_cnn.pt")
print("wrote /kaggle/working/tiny_cnn.pt")


## 学习检查清单

- 你应能回答：如何确认 GPU 可用？训练循环三步是什么？
- 建议：改一个超参重跑一小段，观察 log 变化（比只读代码更有效）。
